<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/DiffractGPT_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[DiffractGPT: Atomic Structure Determination from X‑ray DiffractionPatterns Using a Generative Pretrained Transformer](https://pubs.acs.org/doi/epdf/10.1021/acs.jpclett.4c03137)

GitHub: https://github.com/atomgptlab/atomgpt

YouTube Demo: https://www.youtube.com/watch?v=tyz4MVh2_7k&list=PLjf6vHVv7AoInTVQmfNSMs_12DBXYCcDd&index=2

Author email: kchoudh2@jhu.edu




Following notebook shows examples how to get atmoic structure from powder X-dar diffraction from generative model DiffractGPT and Structure matching methods

In [ ]:
# !pip install -q condacolab
# import condacolab
# condacolab.install()

In [ ]:
%%time
import os
os.chdir('/content')
!rm -rf Software
os.makedirs('/content/Software')
os.chdir('/content/Software')
!pip install uv
if not os.path.exists('atomgpt'):
  !rm -rf atomgpt
  !git clone https://github.com/atomgptlab/atomgpt.git
  os.chdir('atomgpt')
  !pip install -q -e .



In [ ]:
# Check imports
import atomgpt
import os
os.environ.pop('MPLBACKEND', None)  # Remove the invalid backend
import matplotlib
matplotlib.use('Agg')  # Use a compatible backend
import matplotlib.pyplot as plt



In [ ]:
!pip install -q triton==3.2.0

# Inference Example for LaB6


| Argument        | Description |
|----------------|-------------|
| `--model_name` | HuggingFace-compatible model ID for the pretrained DiffractGPT model. Example: `knc6/diffractgpt_mistral_chemical_formula`. |
| `--formula`    | The chemical formula for which the XRD pattern will be generated. Example: `LaB6`. |
| `--dat_path`   | Output path for the generated `.dat` file. The file contains two columns: 2θ (angle) and intensity. |
| `--config_path`| Path to a JSON config file containing generation parameters (use the example file). |


Theoretical XRD data atomgpt/examples/inverse_model_multi/my_data.dat

In [ ]:
os.chdir('/content/Software/atomgpt')

Theoretical XRD data example. Header is chemical formula, the 2theta and intensity values.

In [ ]:
file_content="""LaB6
21.3849 7.2634
30.4238 11.5522
37.4904 4.8924
43.5638 2.4932
49.0228 5.697
54.0619 3.0706
63.3067 1.1376
67.6436 3.2224
71.8493 2.2997
75.9558 1.451
83.9744 0.9538
87.9297 2.0058
99.8122 1.7793
103.8433 1.5299
107.9457 0.6619
112.1457 0.9822
116.4753 2.1496
120.9747 1.1266
130.7195 0.8452
136.154 1.0326
142.1905 4.7341
149.1934 2.7378
"""
with open("my_file.dat","w") as f:
  f.write(file_content)

In [ ]:
%%time
!python atomgpt/inverse_models/inverse_predict.py --model_name knc6/diffractgpt_mistral_chemical_formula --formula LaB6 --dat_path my_file.dat --config_path atomgpt/examples/inverse_model_multi/config.json

## Let's visualize the structure

In [ ]:
!pip install "numpy<2"

In [ ]:
from jarvis.io.vasp.inputs import Poscar
from ase.visualize import view

pos="""System
1.0
4.16 0.0 0.0
0.0 4.16 0.0
0.0 0.0 4.16
La B
1 6
direct
0.0 0.0 0.0 La
0.5 0.5 0.0 B
0.5 0.0 0.5 B
0.0 0.5 0.5 B
0.5 0.0 0.0 B
0.0 0.5 0.0 B
0.0 0.0 0.5 B
"""
atoms = Poscar.from_string(pos).atoms
view(atoms.make_supercell([2,2,2]).ase_converter(),viewer='x3d')

## Structure matching based XRD option using AtomGPT.org API (AGAPI)

In [ ]:
!pip install agapi


In [ ]:
from agapi.client import Agapi
# Go to Website: https://atomgpt.org/
# Navigate to Profile >> Settings >> Account >> API Keys >> Show/Create
# It will look like sk-xxxxxxxxxxxxx paste below
api_key='sk-xxxxxxx'
client = Agapi(api_key=api_key)
r = client.pxrd_query(file_path="my_file.dat")
print(r)